# Pipeline de Classificação LULC — Sentinel-2 + MapBiomas
**Região:** Sapezal (MT) · **Período:** seca 2023 (Jun–Set) · **Classificador:** Random Forest (GEE) + scikit-learn/XGBoost

Todo processamento de imagem é executado **server-side no Google Earth Engine**.  
O Colab/local lida apenas com tabelas de amostras leves e visualizações.

---
## Etapa 1 — Setup do Ambiente

In [ ]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # --- Edite apenas esta linha com a URL do seu repositório GitHub ---
    REPO_URL = 'https://github.com/ferdamarc/LULC_project.git'
    PROJECT_DIR = '/content/LULC_project'

    if not os.path.exists(PROJECT_DIR):
        os.system(f'git clone {REPO_URL} {PROJECT_DIR}')
    else:
        # Re-execuções na mesma sessão: apenas sincroniza mudanças do repo
        os.system(f'git -C {PROJECT_DIR} pull --quiet')

    os.chdir(f'{PROJECT_DIR}/notebooks')
    os.system('pip install -r ../requirements.txt -q')

    sys.path.insert(0, f'{PROJECT_DIR}/src')
    print(f'Colab: projeto carregado de {PROJECT_DIR}')
else:
    # Local: src/ já acessível via kernel lulc-project
    sys.path.insert(0, os.path.abspath('../src'))
    print('Local: usando ambiente Conda lulc-project')

In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import xgboost as xgb
import sys
import os

# Garante que src/ seja importável tanto no Colab quanto localmente
sys.path.insert(0, os.path.abspath('../src'))

print(f"ee: {ee.__version__} | geemap: {geemap.__version__}")

In [ ]:
# Autenticação interativa: abre o fluxo OAuth no browser.
# Necessário apenas na primeira execução por ambiente; o token fica em cache.
ee.Authenticate()

# Substitua pelo ID do seu Google Cloud Project com a Earth Engine API habilitada.
# GCP Console → APIs & Services → Earth Engine API → Enable
GEE_PROJECT = 'seu-projeto-gcp'   # <--- edite aqui

ee.Initialize(project=GEE_PROJECT)
print('GEE inicializado com sucesso.')

---
## Etapa 2 — Definição da AOI e Configuração do Projeto

In [ ]:
# =============================================================================
# Bloco de configuração central — todos os parâmetros do projeto em um lugar.
# Etapas subsequentes leem daqui; não repetir essas constantes nas células abaixo.
# =============================================================================

# --- AOI ---
# Bbox de ~2.400 km² (≈50 km × 50 km) sobre a zona agrícola de Sapezal (MT).
# BBox(west, south, east, north)
AOI = ee.Geometry.BBox(-59.20, -13.85, -58.70, -13.35)

# --- Período temporal ---
START_DATE = '2023-06-01'
END_DATE   = '2023-09-30'

# --- Classes LULC do projeto ---
# Chave: código interno do projeto (1-6)
# 'mapbiomas': lista de códigos MapBiomas Collection 9 que mapeiam para essa classe
# 'color': hex para visualização no mapa
LULC_CLASSES = {
    1: {'name': 'Floresta',                  'mapbiomas': [3],               'color': '#1f7a1f'},
    2: {'name': 'Cerrado/Savana',            'mapbiomas': [4, 12],           'color': '#d4a028'},
    3: {'name': 'Pastagem',                  'mapbiomas': [15],              'color': '#b8af4f'},
    4: {'name': 'Agricultura',               'mapbiomas': [18, 39, 40, 41],  'color': '#f5e642'},
    5: {'name': 'Água',                      'mapbiomas': [33, 31],          'color': '#2d6fd3'},
    6: {'name': 'Solo Exposto/Não Vegetado', 'mapbiomas': [24, 25, 30],      'color': '#c49a6c'},
}

# Lookup inverso: código MapBiomas → código do projeto (usado na Etapa 5)
MB_TO_LULC = {
    mb_code: proj_code
    for proj_code, info in LULC_CLASSES.items()
    for mb_code in info['mapbiomas']
}

# Listas paralelas para uso no GEE (ee.Image.remap espera listas, não dicts)
MB_FROM = list(MB_TO_LULC.keys())    # [3, 4, 12, 15, 18, 39, 40, 41, 33, 31, 24, 25, 30]
MB_TO   = list(MB_TO_LULC.values())  # [1, 2,  2,  3,  4,  4,  4,  4,  5,  5,  6,  6,  6]

print('Configuração carregada.')
print(f'AOI: {AOI.bounds().getInfo()}')  # chamada leve ao GEE para validar o objeto
print(f'Período: {START_DATE} → {END_DATE}')
print(f'Classes: {", ".join(v["name"] for v in LULC_CLASSES.values())}')

In [ ]:
# Calcula a área da AOI server-side (evita .getInfo() em geometrias complexas)
area_km2 = AOI.area(maxError=100).divide(1e6).getInfo()
print(f'Área da AOI: {area_km2:.0f} km²')

In [ ]:
# Visualização interativa da AOI com geemap.
# Em Colab: renderiza como mapa Folium inline.
# Localmente (Jupyter + kernel lulc-project): usa ipyleaflet.
Map = geemap.Map(center=[-13.60, -58.95], zoom=10)
Map.addLayer(AOI, {'color': 'red', 'fillColor': '00000000'}, 'AOI — Sapezal (MT)')

# Adiciona basemap de satélite para contexto visual
Map.add_basemap('SATELLITE')

Map

**Verificação antes de avançar:**
- O contorno vermelho cobre a zona agrícola de Sapezal? (deve mostrar talhões de soja/pastagem no basemap de satélite)
- A área calculada está em ~2.400 km²?
- O GEE não retornou erro de quota ou autenticação?

Se tudo OK, avance para a Etapa 3.

---
## Etapa 3 — Carregamento e Pré-processamento do Sentinel-2
*(implementado na próxima sessão)*

---
## Etapa 4 — Cálculo de Índices Espectrais (NDVI, NDWI, EVI)
*(implementado na próxima sessão)*

---
## Etapa 5 — MapBiomas: Carregamento e Reclassificação
*(implementado na próxima sessão)*

---
## Etapa 6 — Amostragem Estratificada
*(implementado na próxima sessão)*

---
## Etapa 7 — Treinamento do Classificador
*(implementado na próxima sessão)*

---
## Etapa 8 — Validação: Matriz de Confusão, Kappa, F1-score
*(implementado na próxima sessão)*

---
## Etapa 9 — Visualização dos Resultados
*(implementado na próxima sessão)*